# OLBPE-PEMFC: reproduce the results of the paper

Run the cells from top to bottom (**Runtime → Run all** works too).

* Steps 1–3 take about 5 minutes and check that your machine reproduces the published runs exactly.
* Step 4 re-runs **all** experiments from scratch (about 1–1.5 hours on Colab) and compares every run and every table with the published ones.

## 1. Download the code
The code is downloaded from the public repository on GitHub.

In [ ]:
GITHUB_USER = "ahmedsharba-svg"

!git clone -q https://github.com/{GITHUB_USER}/OLBPE-PEMFC.git
%cd OLBPE-PEMFC
!pip install -q -r requirements.txt
!python --version
import numpy, scipy; print('numpy', numpy.__version__, '| scipy', scipy.__version__)

## 2. Self-tests
Checks the model against the published data of Liu & Cheng (2025), the orthogonal array, the statistical tests, the evaluation budget and reproducibility. All five lines must say `[ok]`.

In [ ]:
!python tests/test_all.py

## 3. Quick check (≈ 3 min)
Re-runs runs 1–3 of every algorithm on every stack and compares the RMSE with the stored results. The last line must say **REPRODUCED**.

In [ ]:
!python verify_results.py --quick --runs 3

## 4. Full reproduction (≈ 1–1.5 h)
The published results are first copied to `results_paper/`; then everything is recomputed from scratch into `results/` and compared.

Tip: keep the browser tab open. If the session disconnects, run this cell again — finished runs are kept in `results/*.partial.jsonl` while the session is alive.

In [ ]:
import os, shutil
if not os.path.exists('results_paper'):
    shutil.copytree('results', 'results_paper')          # keep the published results
for f in ['raw_results.json', 'ablation_results.json', 'ablation_results_5k.json']:
    p = os.path.join('results', f)
    if os.path.exists(p):
        os.remove(p)
W = os.cpu_count()
print('workers:', W)
!python run_experiments.py {W}
!python run_ablation.py {W}
!python make_tables_figures.py

## 5. Compare with the paper
Every line must say `[ok]` (run times are excluded because they depend on the computer).

In [ ]:
!python verify_results.py --compare results_paper results

## 6. Look at the tables and figures
Table letters → paper tables: A = Table 5 (parameters), B = Table 6 (statistics), F = Table 7 (FEs to target), C = Table 8 (Wilcoxon), H = Table 9 (residuals), G = Table 10 (identifiability), D/D_5k = Table 11 (ablation), E/E_5k = Table 12 (ablation Wilcoxon).

In [ ]:
import pandas as pd, glob
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
for f in sorted(glob.glob('results/tables/*.csv')):
    print('\n=====', os.path.basename(f))
    display(pd.read_csv(f))

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob('results/figures/*.png')):
    print(os.path.basename(f)); display(Image(f, width=700))

## 7. (optional) Download everything
Creates `my_results.zip` with all regenerated results.

In [ ]:
!zip -qr my_results.zip results
from google.colab import files
files.download('my_results.zip')